# AFIB / Outros / Normal - Pipeline refatorado

Notebook separado do original com foco em:
- split por paciente
- validação consistente
- augmentation online no treino
- early stopping real
- métricas por época no conjunto de validação
- checkpoint do melhor modelo por fold


In [1]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import v2
from PIL import Image
import timm

from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, balanced_accuracy_score, classification_report, confusion_matrix

from tqdm.auto import tqdm
import matplotlib.pyplot as plt


In [2]:
SEED = 42
DATA_CSV = 'dataset_FA_OUTROS_NORMAL_BINARY_HOT_ENCODING.csv'
IMAGE_ROOT = 'dataset_AFIB_Others/data/'
OUT_DIR = Path('saida_refatorada')
OUT_DIR.mkdir(parents=True, exist_ok=True)

N_SPLITS = 5
BATCH_SIZE = 16
EPOCHS = 30
PATIENCE = 6
LR = 1e-4
WEIGHT_DECAY = 1e-4
NUM_CLASSES = 3
IMG_SIZE = 256

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device


device(type='cuda')

In [4]:

import time
ts = int(time.time())
timestamp = str(ts)

if os.getenv("IS_LOCAL"):
    FILE_PATH = 'dataset_FA_OUTROS_NORMAL_BINARY_HOT_ENCODING.csv'

    print("Iniciando o programa em execucao Local")
    print("Parametros de Execucao Local:")
    folds = 5
    epochs = 50
    BATCH_SIZE = 8 # ✅ REDUZIDO de 64 para 32 (evita erro CUDA)
    N_samples = None  # Use None para usar todos os dados
    flg_salvar_modelos = True
    path = '/home/leo/Documents/ecg_classifier/dataset/database_ptbxl/'
    path_out = 'saida_123/saidas/'
    OUTPUT_TEST_PATH = 'OUTPUTTESTES/'
    OUTPUT_MODEL_PATH = 'OUTPUTMODELS/'

    print(f"  Numero de folds para K-Fold Cross Validation: {folds}")

    print(f"  Tamanho do batch para treinamento: {BATCH_SIZE}")

    print(f"  Numero de epocas para treinamento: {epochs}")
else:
    from google.colab import drive
    drive.mount('/content/drive')
    folds = 5
    epochs = 50
    BATCH_SIZE = 64 # ✅ REDUZIDO de 64 para 32 (evita erro CUDA)
    N_samples = None  # Use None para usar todos os dados
    flg_salvar_modelos = True
    os.system('cp  "/content/drive/MyDrive/dataset_AFIB_Others/database_ptbxl.zip" "/content/dataset.zip"')
    os.system('unzip -oq dataset.zip')
    FILE_PATH = '/content/drive/MyDrive/dataset_AFIB_Others/dataset_FA_OUTROS_NORMAL_BINARY_HOT_ENCODING.csv'
    IMAGE_ROOT = '/content/content/drive/MyDrive/dataset_AFIB_Others/database_ptbxl/'
    path_out = f'/content/drive/MyDrive/dataset_AFIB_Others/saida_{timestamp}/saidas/'
    OUTPUT_TEST_PATH = f'/content/drive/MyDrive/dataset_AFIB_Others/OUTPUTTESTES/'
    OUTPUT_MODEL_PATH = f'/content/drive/MyDrive/dataset_AFIB_Others/OUTPUTMODELS/'


Mounted at /content/drive


In [5]:
def load_data(csv_path):
    df = pd.read_csv(csv_path)
    required = {'patient_id', 'path', 'NORMAL', 'Other', 'AFIB'}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f'Colunas obrigatorias ausentes: {missing}')
    df = df.copy()
    df['rotulo'] = df[['NORMAL', 'Other', 'AFIB']].astype(int).astype(str).agg(''.join, axis=1)
    df['label_idx'] = df[['NORMAL', 'Other', 'AFIB']].astype(int).idxmax(axis=1).map({'NORMAL': 0, 'Other': 1, 'AFIB': 2})
    return df

data = load_data(FILE_PATH)
data.head()


,patient_id,path,age,sex,height,weight,AFIB,NORMAL,Other,rotulo,label_idx
0,15709.0,00001_lr-0.png,56.0,1,NaN,63.0,False,True,False,100,0
1,13243.0,00002_lr-0.png,19.0,0,NaN,70.0,False,False,True,010,1
2,20372.0,00003_lr-0.png,37.0,1,NaN,69.0,False,True,False,100,0
3,17014.0,00004_lr-0.png,24.0,0,NaN,82.0,False,True,False,100,0
4,17448.0,00005_lr-0.png,19.0,1,NaN,70.0,False,True,False,100,0


In [7]:
class ECGDataset(Dataset):
    def __init__(self, df, image_root, transform=None, train=False):
        self.df = df.reset_index(drop=True).copy()
        self.image_root = Path(image_root)
        self.transform = transform
        self.train = train

    def __len__(self):
        return len(self.df)

    def _load_image(self, rel_path):
        img = Image.open(self.image_root / rel_path).convert('RGB')
        return img

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = self._load_image(row['path'])
        if self.transform is not None:
            img = self.transform(img)
        label = torch.tensor(row['label_idx'], dtype=torch.long)
        return img, label

train_transform = v2.Compose([
    v2.Resize((IMG_SIZE, IMG_SIZE)),
    v2.RandomRotation(degrees=10),
    v2.RandomAffine(degrees=0, translate=(0.03, 0.03), scale=(0.97, 1.03)),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

val_transform = v2.Compose([
    v2.Resize((IMG_SIZE, IMG_SIZE)),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])


In [8]:
def build_model(num_classes=3, backbone='resnet50d.ra4_e3600_r224_in1k', pretrained=True, freeze_backbone=True):
    model = timm.create_model(backbone, pretrained=pretrained, num_classes=num_classes)
    if freeze_backbone:
        for name, param in model.named_parameters():
            if 'fc' not in name and 'classifier' not in name and 'head' not in name:
                param.requires_grad = False
    return model

def class_weights_from_df(df):
    counts = df['label_idx'].value_counts().sort_index()
    weights = 1.0 / counts
    weights = weights / weights.sum() * len(weights)
    return torch.tensor(weights.values, dtype=torch.float32, device=device)


In [9]:
def compute_metrics(y_true, y_pred):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'precision_macro': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall_macro': recall_score(y_true, y_pred, average='macro', zero_division=0),
    }

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    y_true, y_pred = [], []
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        logits = model(x)
        loss = criterion(logits, y)
        total_loss += loss.item() * x.size(0)
        y_true.extend(y.cpu().numpy().tolist())
        y_pred.extend(torch.argmax(logits, dim=1).cpu().numpy().tolist())
    metrics = compute_metrics(y_true, y_pred)
    metrics['loss'] = total_loss / len(loader.dataset)
    return metrics, np.array(y_true), np.array(y_pred)


In [10]:
def train_one_fold(train_df, val_df, fold_idx):
    train_ds = ECGDataset(train_df, IMAGE_ROOT, transform=train_transform, train=True)
    val_ds = ECGDataset(val_df, IMAGE_ROOT, transform=val_transform, train=False)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    model = build_model(num_classes=NUM_CLASSES, freeze_backbone=True).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights_from_df(train_df), label_smoothing=0.05)
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

    best_val = float('inf')
    best_state = None
    patience_count = 0
    history = []

    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        for x, y in tqdm(train_loader, desc=f'Fold {fold_idx} | Epoch {epoch+1}/{EPOCHS}'):
            x = x.to(device)
            y = y.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            running_loss += loss.item() * x.size(0)

        train_loss = running_loss / len(train_loader.dataset)
        val_metrics, y_true, y_pred = evaluate(model, val_loader, criterion)
        scheduler.step(val_metrics['loss'])

        row = {'epoch': epoch + 1, 'train_loss': train_loss, **val_metrics}
        history.append(row)

        if val_metrics['loss'] < best_val - 1e-4:
            best_val = val_metrics['loss']
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_count = 0
            torch.save(best_state, OUT_DIR / f'best_fold_{fold_idx}.pth')
        else:
            patience_count += 1

        print(row)
        if patience_count >= PATIENCE:
            print(f'Early stopping no fold {fold_idx} na epoca {epoch+1}')
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, pd.DataFrame(history)


In [11]:
def run_cv(df):
    gkf = GroupKFold(n_splits=N_SPLITS)
    fold_results = []
    for fold_idx, (train_idx, val_idx) in enumerate(gkf.split(df, groups=df['patient_id'])):
        train_df = df.iloc[train_idx].reset_index(drop=True)
        val_df = df.iloc[val_idx].reset_index(drop=True)

        assert set(train_df['patient_id']).isdisjoint(set(val_df['patient_id']))

        model, history = train_one_fold(train_df, val_df, fold_idx)
        criterion = nn.CrossEntropyLoss()
        val_loader = DataLoader(ECGDataset(val_df, IMAGE_ROOT, transform=val_transform), batch_size=BATCH_SIZE, shuffle=False)
        final_metrics, y_true, y_pred = evaluate(model, val_loader, criterion)
        cm = confusion_matrix(y_true, y_pred)

        history.to_csv(OUT_DIR / f'history_fold_{fold_idx}.csv', index=False)
        np.save(OUT_DIR / f'confusion_matrix_fold_{fold_idx}.npy', cm)
        pd.DataFrame([final_metrics]).to_csv(OUT_DIR / f'metrics_fold_{fold_idx}.csv', index=False)

        fold_results.append(final_metrics)
        print(classification_report(y_true, y_pred, digits=4, zero_division=0))

    return pd.DataFrame(fold_results)


In [12]:
# Split final opcional: teste intocado antes do CV
test_df = None
trainval_df, test_df = train_test_split(
    data,
    test_size=0.2,
    random_state=SEED,
    stratify=data['label_idx']
)

cv_results = run_cv(trainval_df.reset_index(drop=True))
cv_results


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


model.safetensors:   0%|          | 0.00/103M [00:00<?, ?B/s]

Fold 0 | Epoch 1/30:   0%|          | 0/218 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x798436473740>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^Exception ignored in: ^<function _MultiProcessingDataLoaderIter.__del__ at 0x798436473740>^
^Traceback (most recent call last):
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
^^    ^self._shutdown_workers()^
^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
^    ^if w.is_alive():^
^ ^ ^ ^ ^ ^ ^ 

{'epoch': 1, 'train_loss': 1.1745624777305324, 'accuracy': 0.16083715596330275, 'balanced_accuracy': np.float64(0.39824988428581937), 'f1_macro': 0.1747513249598168, 'precision_macro': 0.36209275473035896, 'recall_macro': 0.39824988428581937, 'loss': 1.1608859661522262}


...

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x798436473740>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x798436473740>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

{'epoch': 2, 'train_loss': 1.163256871726776, 'accuracy': 0.2006880733944954, 'balanced_accuracy': np.float64(0.42657904417093023), 'f1_macro': 0.20777880481082087, 'precision_macro': 0.3686358069991203, 'recall_macro': 0.42657904417093023, 'loss': 1.1513057629996484}


Fold 0 | Epoch 3/30:   0%|          | 0/218 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x798436473740>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^Exception ignored in: ^<function _MultiProcessingDataLoaderIter.__del__ at 0x798436473740>^
^Traceback (most recent call last):
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
^    ^self._shutdown_workers()^
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
^^    ^if w.is_alive():^
^ ^ ^ ^ ^^ 

KeyboardInterrupt: 

In [ ]:
summary = cv_results.agg(['mean', 'std'])
summary


## Próximos passos

- Trocar `GroupKFold` por um split estratificado por grupo, se a distribuição de classes por paciente estiver muito desigual.
- Testar `freeze_backbone=False` apenas após a fase inicial de head training.
- Avaliar se o backbone de ImageNet é o melhor ponto de partida para esse domínio.
